In [23]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.linear_model import LinearRegression
import yfinance as yf

In [24]:
end = dt.date.today()
start = dt.date.today() - dt.timedelta(days=365*5)
ticker = ['HDFCBANK.NS','^NSEI','^INDIAVIX','^NSEBANK']
df = yf.download(tickers=ticker,start=start,end=end,auto_adjust=True)['Close']

# Dropping Nans
df[df.isna().any(axis=1)]
df.dropna(inplace=True)

# Features Engineering
df['hdfc_returns'] = np.log(df['HDFCBANK.NS']/df['HDFCBANK.NS'].shift(1))
df['hdfc_lagged'] = df['hdfc_returns'].shift(1)
df['nse_returns'] = np.log(df['^NSEI']/df['^NSEI'].shift(1))
df['bank_returns'] = np.log(df['^NSEBANK']/df['^NSEBANK'].shift(1))
df['volatility'] = df['hdfc_returns'].rolling(20).std()
df['momentum'] = df['HDFCBANK.NS'].pct_change(5)

# Cleaning
df.drop(columns=['^NSEBANK','^NSEI','HDFCBANK.NS'],inplace=True)
df.dropna(inplace=True)
df.columns = df.columns.str.lower()
df.rename(columns = {'^indiavix':'vix'},inplace=True)
columns = ['hdfc_returns','hdfc_lagged','nse_returns','bank_returns','volatility','momentum','vix']
df = df[[col for col in columns if col in df.columns]]

# ADF test

from statsmodels.tsa.stattools import adfuller
def adf_test(series, name=""):
    result = adfuller(series.dropna())

    print(f"\nADF Test → {name}")
    print(f"ADF Statistic: {result[0]:.2f}")
    print(f"p-value: {result[1]:.2f}")

    if result[1] < 0.05:
        print("→ Stationary ✅")
    else:
        print("→ Non-stationary ❌")

for col in df.columns:
    adf_test(df[col], col)


[*********************100%***********************]  4 of 4 completed



ADF Test → hdfc_returns
ADF Statistic: -33.90
p-value: 0.00
→ Stationary ✅

ADF Test → hdfc_lagged
ADF Statistic: -33.66
p-value: 0.00
→ Stationary ✅

ADF Test → nse_returns
ADF Statistic: -35.31
p-value: 0.00
→ Stationary ✅

ADF Test → bank_returns
ADF Statistic: -34.74
p-value: 0.00
→ Stationary ✅

ADF Test → volatility
ADF Statistic: -3.39
p-value: 0.01
→ Stationary ✅

ADF Test → momentum
ADF Statistic: -6.77
p-value: 0.00
→ Stationary ✅

ADF Test → vix
ADF Statistic: -2.68
p-value: 0.08
→ Non-stationary ❌


In [25]:
# Splitting Data
train = int(len(df) * 0.8)
train_df = df.iloc[:train]
test_df = df.iloc[train:]

# Define X and y
x_train = train_df.drop(columns=['vix','hdfc_returns','volatility','nse_returns'])
y_train = train_df['hdfc_returns']

# Train model using SKlearn
model = LinearRegression()
model = model.fit(x_train, y_train)

In [26]:
# Training Model using Statsmodels for statistical analysis
import statsmodels.api as sm

x_train_sm = sm.add_constant(x_train)
model_sm = sm.OLS(y_train, x_train_sm).fit()

# Statsitical Analysis
# % of target variation is explained by model
print("R²:", model_sm.rsquared)

# To detect useless feature
print("Adj R²:", model_sm.rsquared_adj)

# to test model is useful or not
print("Prob(F):", model_sm.f_pvalue)

# Impact of each feature on target
print("\nCoefficients:")
print(model_sm.params)

#significance of each featurex
print("\np-values:")
print(model_sm.pvalues)


R²: 0.6354393238848826
Adj R²: 0.6342976933751274
Prob(F): 2.4126993264009758e-209

Coefficients:
const          -0.000291
hdfc_lagged    -0.092117
bank_returns    0.833790
momentum        0.104857
dtype: float64

p-values:
const            2.681321e-01
hdfc_lagged      4.097079e-05
bank_returns    7.666358e-152
momentum         9.117199e-22
dtype: float64


In [27]:
y_pred_train = model.predict(x_train)

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Train errors
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train)

print("\nTrain RMSE:", rmse_train)
print("Train MAE:", mae_train)

X_test = test_df.drop(columns=['vix','hdfc_returns','volatility','nse_returns'])
y_test = test_df['hdfc_returns']

y_pred_test = model.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print("\nTest RMSE:", rmse_test)
print("Test MAE:", mae_test)


Train RMSE: 0.008123476254171412
Train MAE: 0.005822155793067171

Test RMSE: 0.0062561554038005245
Test MAE: 0.004759038826709173
